In [1]:
## Cell 1 · Imports

import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

df = pd.read_csv("csv/00_base_data.csv")
print(f"pandas {pd.__version__}")
print(f"Loaded {len(df)} records")

PLUTO_CSV = "../ramy/NYC_pluto_25v4_csv/pluto_25v4.csv"

pandas 3.0.2
Loaded 2915 records


In [2]:
## Cell 2 · Load PLUTO & Match Building Age

print("Loading PLUTO data...")
pluto = pd.read_csv(PLUTO_CSV, low_memory=False)
pluto_mn = pluto[pluto["borough"] == "MN"].dropna(subset=["latitude", "longitude"]).copy()
print(f"  {len(pluto_mn)} Manhattan lots loaded")

# Build spatial index
pluto_coords = np.radians(pluto_mn[["latitude", "longitude"]].values)
tree = BallTree(pluto_coords, metric="haversine")

# Match each shop to nearest lot
shop_coords = np.radians(df[["lat", "lon"]].values)
distances, indices = tree.query(shop_coords, k=1)

# Attach yearbuilt
df["year_built"] = pluto_mn.iloc[indices.flatten()]["yearbuilt"].values
df["year_built"] = df["year_built"].replace(0, np.nan)

print(f"\nyear_built fill : {df['year_built'].notna().sum()}/{len(df)}")
print(f"  Mean : {df['year_built'].mean():.0f}")
print(f"  Min  : {df['year_built'].min():.0f}")
print(f"  Max  : {df['year_built'].max():.0f}")

Loading PLUTO data...
  42116 Manhattan lots loaded

year_built fill : 2832/2915
  Mean : 1939
  Min  : 914
  Max  : 2025


In [3]:
## Cell 3 · Save

df_out = df[["osm_id", "year_built"]]
df_out.to_csv("csv/08_building_age.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/08_building_age.csv")
print(df_out.describe().round(0))

Saved 2915 records to csv/08_building_age.csv
             osm_id  year_built
count  2.915000e+03      2832.0
mean   6.217806e+09      1939.0
std    4.166958e+09        43.0
min    8.539400e+07       914.0
25%    2.709934e+09      1920.0
50%    5.726058e+09      1926.0
75%    1.023030e+10      1962.0
max    1.380607e+10      2025.0
